# 06_feature_eng.ipynb

## Sprint 2 - PB-08: Ingeniería de características

En este notebook se realiza la etapa de **feature engineering** sobre el dataset limpio generado en PB-07.

### Objetivo
Construir nuevas variables y transformar variables existentes para enriquecer la información disponible para el modelado posterior.

### Alcance de esta etapa
En este notebook se desarrollan las siguientes tareas:
- creación de features derivadas
- extracción de componentes de fecha
- binning de variables numéricas
- creación de interacciones entre variables
- transformaciones para reducir asimetría
- selección preliminar de variables

### Input
- `data/interim/cleaned_data.csv`

### Output
- `data/interim/featured_data.csv`

## 1. Importación de librerías
Se importan las librerías necesarias para manipulación de datos, creación de variables nuevas y validaciones básicas.

In [ ]:
import os
import numpy as np
import pandas as pd

Shape inicial: (72983, 53)


,IsBadBuy,PurchDate,VehYear,VehicleAge,Make,Model,Trim,SubModel,WheelTypeID,VehOdo,...,Color_OTHER,Color_PURPLE,Color_RED,Color_SILVER,Color_WHITE,Color_YELLOW,Auction_MANHEIM,Auction_OTHER,WheelType_Covers,WheelType_Special
0,0,1970-01-01 00:00:01.260144000,2006.0,-36.0,0.161389,0.084158,0.112936,0.111111,1.0,89046.0,...,False,False,True,False,False,False,False,False,False,False
1,0,1970-01-01 00:00:01.260144000,2004.0,-34.0,0.103237,0.116258,0.115339,0.122195,1.0,93593.0,...,False,False,False,False,True,False,False,False,False,False
2,0,1970-01-01 00:00:01.260144000,2005.0,-35.0,0.103237,0.097606,0.098824,0.028336,2.0,73807.0,...,False,False,False,False,False,False,False,False,True,False
3,0,1970-01-01 00:00:01.260144000,2004.0,-34.0,0.103237,0.234043,0.098824,0.122278,1.0,65617.0,...,False,False,False,True,False,False,False,False,False,False
4,0,1970-01-01 00:00:01.260144000,2005.0,-35.0,0.154091,0.171617,0.154472,0.160377,2.0,69367.0,...,False,False,False,True,False,False,False,False,True,False


## 2. Carga del dataset limpio
Se carga el dataset resultante de la etapa de limpieza (`PB-07`), el cual será utilizado como base para construir nuevas variables.
Para evitar modificar directamente el dataset original de entrada, se trabaja sobre una copia.

In [ ]:
df = pd.read_csv("/Users/alexandralozano/dp261-g1/data/interim/cleaned_data.csv")
df_fe = df.copy()

print("Shape inicial:", df_fe.shape)
df_fe.head()

## 3. Revisión inicial de variables

Antes de crear nuevas features, se revisan:
- dimensiones del dataset
- nombres de columnas
- tipos de datos

Esto permite identificar qué variables son candidatas para:
- ratios
- interacciones
- transformaciones
- extracción de fechas
- binning

In [5]:
# Revisar forma general del dataset
print("Shape inicial:", df_fe.shape)

# Revisar columnas disponibles
print(df_fe.columns.tolist())

# Revisar tipos de datos
print(df_fe.dtypes)

Shape inicial: (72983, 53)
['IsBadBuy', 'PurchDate', 'VehYear', 'VehicleAge', 'Make', 'Model', 'Trim', 'SubModel', 'WheelTypeID', 'VehOdo', 'Nationality', 'Size', 'TopThreeAmericanName', 'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice', 'PRIMEUNIT', 'AUCGUART', 'BYRNO', 'VNZIP1', 'VNST', 'VehBCost', 'IsOnlineSale', 'WarrantyCost', 'year', 'month', 'edad_calculada', 'Transmission_MANUAL', 'Transmission_Manual', 'Color_BLACK', 'Color_BLUE', 'Color_BROWN', 'Color_GOLD', 'Color_GREEN', 'Color_GREY', 'Color_MAROON', 'Color_NOT AVAIL', 'Color_ORANGE', 'Color_OTHER', 'Color_PURPLE', 'Color_RED', 'Color_SILVER', 'Color_WHITE', 'Color_YELLOW', 'Auction_MANHEIM', 'Auction_OTHER', 'WheelType_Covers', 'WheelType_Special']
IsBadBuy                               int64
PurchDate         

## 4. Creación de features derivadas

En esta sección se construyen nuevas variables a partir de columnas existentes.

### Criterio
Se crean features derivadas cuando una combinación entre variables puede capturar mejor:
- relaciones de proporción
- diferencias relevantes
- señales de desgaste, costo relativo o variación de precio

### Ejemplos de lógica utilizada
- ratios: relación entre dos variables
- spreads: diferencia entre dos precios o medidas
- métricas relativas: valor por unidad o por antigüedad

In [6]:
# Crear variable de desgaste relativo: kilometraje por año de antigüedad
# Hipótesis: un auto con más kilometraje relativo a su edad podría tener mayor riesgo
if {"VehOdo", "VehicleAge"}.issubset(df_fe.columns):
    df_fe["odo_per_year"] = df_fe["VehOdo"] / (df_fe["VehicleAge"] + 1)

# Crear ratio de garantía sobre costo del vehículo
# Hipótesis: un mayor costo de garantía relativo al precio puede reflejar mayor riesgo o menor calidad
if {"WarrantyCost", "VehBCost"}.issubset(df_fe.columns):
    df_fe["warranty_cost_ratio"] = df_fe["WarrantyCost"] / (df_fe["VehBCost"] + 1)

# Comparar costo de compra con precio de referencia de adquisición
# Hipótesis: diferencias frente al valor de referencia pueden capturar oportunidades o riesgo
if {"VehBCost", "MMRAcquisitionAuctionAveragePrice"}.issubset(df_fe.columns):
    df_fe["cost_vs_acq_auction_avg_ratio"] = (
        df_fe["VehBCost"] / (df_fe["MMRAcquisitionAuctionAveragePrice"] + 1)
    )

## 5. Extracción de componentes de fecha

Cuando el dataset contiene variables temporales, es útil descomponerlas en componentes más simples.

### Objetivo
Capturar posibles patrones de:
- estacionalidad
- comportamiento por mes
- comportamiento por día de la semana
- efectos de calendario

La fecha original puede contener más información al separarla en variables derivadas.

In [7]:
# Convertir la fecha a formato datetime para poder extraer componentes temporales
if "PurchDate" in df_fe.columns:
    df_fe["PurchDate"] = pd.to_datetime(df_fe["PurchDate"], errors="coerce")

    # Año de compra
    df_fe["purchase_year"] = df_fe["PurchDate"].dt.year

    # Mes de compra
    df_fe["purchase_month"] = df_fe["PurchDate"].dt.month

    # Día del mes
    df_fe["purchase_day"] = df_fe["PurchDate"].dt.day

    # Día de la semana
    df_fe["purchase_dow"] = df_fe["PurchDate"].dt.dayofweek

## 6. Binning de variables numéricas

En esta sección se agrupan algunas variables numéricas en categorías.

### Objetivo
El binning permite:
- simplificar la interpretación
- capturar efectos no lineales
- reducir sensibilidad a valores individuales
- transformar variables continuas en grupos con significado práctico

### Criterio
Se aplica binning a variables donde tiene sentido agrupar observaciones por tramos.

In [8]:
# Agrupar la antigüedad del vehículo en categorías
# Hipótesis: el riesgo puede comportarse distinto por tramos de edad
if "VehicleAge" in df_fe.columns:
    df_fe["vehicle_age_group"] = pd.cut(
        df_fe["VehicleAge"],
        bins=[-1, 2, 5, 10, 30],
        labels=["nuevo", "semi_nuevo", "usado", "antiguo"]
    )

# Agrupar el odómetro por cuantiles
# Hipótesis: niveles relativos de kilometraje pueden ser más interpretables que el valor exacto
if "VehOdo" in df_fe.columns:
    df_fe["odometer_group"] = pd.qcut(
        df_fe["VehOdo"],
        q=4,
        labels=["bajo", "medio_bajo", "medio_alto", "alto"],
        duplicates="drop"
    )

## 7. Creación de interacciones entre variables

Las interacciones permiten capturar efectos conjuntos entre variables que, por separado, podrían no reflejar completamente un patrón.

### Objetivo
Modelar relaciones combinadas entre variables relevantes.

### Ejemplo conceptual
El efecto de la antigüedad puede cambiar dependiendo del kilometraje, por lo que una interacción entre ambas podría aportar más información que cada variable por separado.

In [9]:
# Interacción entre antigüedad y kilometraje
# Hipótesis: el desgaste del vehículo puede depender del efecto combinado de ambas variables
if {"VehicleAge", "VehOdo"}.issubset(df_fe.columns):
    df_fe["age_x_odo"] = df_fe["VehicleAge"] * df_fe["VehOdo"]

# Interacción entre costo y garantía
# Hipótesis: puede capturar situaciones donde el costo del vehículo y el costo de garantía se refuerzan entre sí
if {"VehBCost", "WarrantyCost"}.issubset(df_fe.columns):
    df_fe["cost_x_warranty"] = df_fe["VehBCost"] * df_fe["WarrantyCost"]

## 8. Transformaciones para reducir asimetría

Algunas variables numéricas pueden presentar distribuciones altamente sesgadas.

### Objetivo
Reducir asimetría y mejorar la estabilidad de las variables.

### Transformación aplicada
Se utiliza `log1p` en variables no negativas, ya que:
- reduce el impacto de valores extremos
- conserva el orden de los datos
- permite trabajar con ceros

> En etapas posteriores, otras transformaciones más complejas pueden integrarse directamente en el pipeline.

In [10]:
# Variables candidatas a transformación logarítmica
skew_candidates = ["VehOdo", "VehBCost", "WarrantyCost"]

for col in skew_candidates:
    if col in df_fe.columns and (df_fe[col] >= 0).all():
        # Crear versión transformada con log1p
        df_fe[f"{col}_log1p"] = np.log1p(df_fe[col])

## 9. Selección preliminar de variables

En esta sección se realiza una revisión inicial para identificar:
- variables con varianza nula
- variables altamente redundantes por correlación

### Objetivo
Reducir ruido y evitar mantener variables que no aportan información o que duplican demasiado la señal de otras.

> Esta selección es preliminar y se documenta antes de eliminar variables de manera definitiva.

In [11]:
# Detectar columnas constantes
constant_cols = [c for c in df_fe.columns if df_fe[c].nunique(dropna=False) <= 1]

# Eliminar columnas sin variación
df_fe = df_fe.drop(columns=constant_cols)

# Analizar correlación entre variables numéricas
num_df = df_fe.select_dtypes(include=["number"]).drop(columns=["IsBadBuy"], errors="ignore")
corr_matrix = num_df.corr().abs()

# Quedarse con la parte superior de la matriz para evitar duplicados
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Detectar variables muy correlacionadas
high_corr_cols = [column for column in upper.columns if any(upper[column] > 0.95)]

print("Columnas constantes eliminadas:", constant_cols)
print("Columnas altamente correlacionadas detectadas:", high_corr_cols)

Columnas constantes eliminadas: ['year', 'month', 'purchase_year', 'purchase_month', 'purchase_day', 'purchase_dow', 'vehicle_age_group']
Columnas altamente correlacionadas detectadas: ['VehicleAge', 'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailCleanPrice', 'edad_calculada', 'odo_per_year', 'age_x_odo', 'VehOdo_log1p', 'VehBCost_log1p']


## 10. Documentación de features creadas

A continuación se documentan las principales variables nuevas creadas en esta etapa.

Para cada feature se registra:
- nombre
- cálculo realizado
- hipótesis de utilidad

Esta documentación permite mantener trazabilidad y justificar el valor potencial de cada nueva variable.

In [12]:
# Tabla resumen de features nuevas creadas
feature_docs = [
    {
        "feature": "odo_per_year",
        "calculo": "VehOdo / (VehicleAge + 1)",
        "hipotesis": "captura desgaste relativo del vehículo"
    },
    {
        "feature": "warranty_cost_ratio",
        "calculo": "WarrantyCost / VehBCost",
        "hipotesis": "aproxima el costo de garantía relativo al valor del vehículo"
    },
    {
        "feature": "purchase_month",
        "calculo": "mes extraído de PurchDate",
        "hipotesis": "captura posible estacionalidad en la compra"
    },
    {
        "feature": "age_x_odo",
        "calculo": "VehicleAge * VehOdo",
        "hipotesis": "captura el efecto conjunto entre antigüedad y kilometraje"
    }
]

feature_docs_df = pd.DataFrame(feature_docs)
feature_docs_df

,feature,calculo,hipotesis
0,odo_per_year,VehOdo / (VehicleAge + 1),captura desgaste relativo del vehículo
1,warranty_cost_ratio,WarrantyCost / VehBCost,aproxima el costo de garantía relativo al valo...
2,purchase_month,mes extraído de PurchDate,captura posible estacionalidad en la compra
3,age_x_odo,VehicleAge * VehOdo,captura el efecto conjunto entre antigüedad y ...


## 11. Verificación final del dataset enriquecido

Se revisa la forma final del dataset y se identifican las nuevas columnas generadas respecto al dataset de entrada.

Esto permite validar que la etapa de feature engineering produjo efectivamente nuevas variables útiles para el siguiente paso del Sprint 2.

In [13]:
# Comparar shape inicial vs final
print("Shape inicial:", df.shape)
print("Shape final:", df_fe.shape)

# Identificar columnas nuevas
new_cols = [c for c in df_fe.columns if c not in df.columns]
print("Nuevas columnas creadas:")
print(new_cols)

Shape inicial: (72983, 53)
Shape final: (72983, 60)
Nuevas columnas creadas:
['odo_per_year', 'warranty_cost_ratio', 'cost_vs_acq_auction_avg_ratio', 'odometer_group', 'age_x_odo', 'cost_x_warranty', 'VehOdo_log1p', 'VehBCost_log1p', 'WarrantyCost_log1p']


## 12. Guardado del dataset de salida

Finalmente, se guarda el dataset resultante de la etapa de feature engineering en la carpeta `data/interim/`.

Este archivo será utilizado como input de la siguiente etapa del Sprint 2.

In [15]:
# Guardar dataset enriquecido
df_fe.to_csv("/Users/alexandralozano/dp261-g1/data/interim/featured_data.csv", index=False)

print("Archivo guardado en data/interim/featured_data.csv")

Archivo guardado en data/interim/featured_data.csv
